# GroupBy and Aggregation in Pandas

This notebook demonstrates grouping, multiple metrics, named aggregation, transformations, filtering, missing keys, and reporting.

In [1]:
import pandas as pd

sales = pd.DataFrame({
    'region': ['East', 'West', 'East', 'North', 'West', 'East', 'North'],
    'product': ['A', 'A', 'B', 'B', 'C', 'A', 'C'],
    'salesperson': ['Mira', 'Dev', 'Mira', 'Arun', 'Dev', 'Zoya', 'Arun'],
    'units': [10, 7, 5, 9, 4, 8, 6],
    'revenue': [1000, 700, 750, 1350, 600, 800, 900]
})
sales

,region,product,salesperson,units,revenue
0,East,A,Mira,10,1000
1,West,A,Dev,7,700
2,East,B,Mira,5,750
3,North,B,Arun,9,1350
4,West,C,Dev,4,600
5,East,A,Zoya,8,800
6,North,C,Arun,6,900


## 1. Group by one key

In [2]:
sales.groupby('region')['revenue'].sum()

region
East     2550
North    2250
West     1300
Name: revenue, dtype: int64

In [3]:
sales.groupby('product')['units'].mean()

product
A    8.333333
B    7.000000
C    5.000000
Name: units, dtype: float64

## 2. Multiple aggregations

In [4]:
sales.groupby('region')['revenue'].agg(['sum', 'mean', 'min', 'max', 'count'])

,sum,mean,min,max,count
region,,,,,
East,2550,850.0,750,1000,3
North,2250,1125.0,900,1350,2
West,1300,650.0,600,700,2


## 3. Named aggregation for reports

In [5]:
summary = sales.groupby('region', as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    average_revenue=('revenue', 'mean'),
    total_units=('units', 'sum'),
    products=('product', 'nunique'),
    transactions=('revenue', 'size')
)
summary.sort_values('total_revenue', ascending=False)

,region,total_revenue,average_revenue,total_units,products,transactions
0,East,2550,850.0,23,2,3
1,North,2250,1125.0,15,2,2
2,West,1300,650.0,11,2,2


## 4. Group by multiple keys

In [6]:
sales.groupby(['region', 'product'], as_index=False).agg(
    total_revenue=('revenue', 'sum'),
    total_units=('units', 'sum'),
    transactions=('revenue', 'size')
)

,region,product,total_revenue,total_units,transactions
0,East,A,1800,18,2
1,East,B,750,5,1
2,North,B,1350,9,1
3,North,C,900,6,1
4,West,A,700,7,1
5,West,C,600,4,1


## 5. size() versus count()

In [7]:
with_missing = pd.DataFrame({'region': ['East', 'East', 'West'], 'revenue': [100, None, 200]})
print('Rows per region:')
display(with_missing.groupby('region').size())
print('Non-missing revenue per region:')
display(with_missing.groupby('region')['revenue'].count())

Rows per region:


region
East    2
West    1
dtype: int64

Non-missing revenue per region:


region
East    1
West    1
Name: revenue, dtype: int64

## 6. Transform group summaries back onto rows

In [8]:
with_share = sales.copy()
with_share['region_total'] = with_share.groupby('region')['revenue'].transform('sum')
with_share['share_of_region'] = with_share['revenue'] / with_share['region_total']
with_share

,region,product,salesperson,units,revenue,region_total,share_of_region
0,East,A,Mira,10,1000,2550,0.392157
1,West,A,Dev,7,700,1300,0.538462
2,East,B,Mira,5,750,2550,0.294118
3,North,B,Arun,9,1350,2250,0.600000
4,West,C,Dev,4,600,1300,0.461538
5,East,A,Zoya,8,800,2550,0.313725
6,North,C,Arun,6,900,2250,0.400000


## 7. Filter groups by a condition

In [9]:
sales.groupby('region').filter(lambda group: group['revenue'].sum() >= 1500)

,region,product,salesperson,units,revenue
0,East,A,Mira,10,1000
2,East,B,Mira,5,750
3,North,B,Arun,9,1350
5,East,A,Zoya,8,800
6,North,C,Arun,6,900


## 8. Missing group keys and group ordering

In [10]:
missing_region = pd.DataFrame({'region': ['East', None, 'East', 'West'], 'revenue': [10, 20, 30, 40]})
print('Default (missing key omitted):')
display(missing_region.groupby('region')['revenue'].sum())
print('Include missing key as a group:')
display(missing_region.groupby('region', dropna=False)['revenue'].sum())

Default (missing key omitted):


region
East    40
West    40
Name: revenue, dtype: int64

Include missing key as a group:


region
East    40
West    40
NaN     20
Name: revenue, dtype: int64

## 9. Custom aggregation and total validation

In [11]:
revenue_range = sales.groupby('region')['revenue'].agg(lambda s: s.max() - s.min())
print('Revenue range by region:')
display(revenue_range)
regional_total = sales.groupby('region')['revenue'].sum().sum()
overall_total = sales['revenue'].sum()
assert regional_total == overall_total
print('Regional totals match overall total:', regional_total == overall_total)

Revenue range by region:


region
East     250
North    450
West     100
Name: revenue, dtype: int64

Regional totals match overall total: True


## Key takeaways

- `groupby()` follows the split-apply-combine pattern.
- `agg()` summarizes groups, while `transform()` keeps the original row count.
- Named aggregation makes report columns easier to read.
- Handle missing grouping keys deliberately.